<a href="https://colab.research.google.com/github/cbonnin88/ReviewMiner/blob/main/ReviewMiner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install streamlit
!pip install pyngrok -q
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=bd11b4b2e78cb476090f47e15903f79b9b479a70ad92a749e9df200e24509da2
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [44]:
%%writefile app.py
import streamlit as st
import pandas as pd
from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from fpdf import FPDF
import base64
import numpy as np


# --- Page Setup ---
st.set_page_config(page_title='Netflix PM Qualitative Analysis',layout='wide')
st.title('📺 Netflix Product Manager Dashboard')
st.write('Extracting themes, sentiment, and actionable backlog items from user reviews.')

# --- File Uploader ---
uploaded_file = st.file_uploader('Upload netlfix_review.csv', type='csv')

if uploaded_file is not None:
  df_netflix = pd.read_csv(uploaded_file)
  df_netflix = df_netflix.dropna(subset=['content'])

  st.write(f'**Total Reviews Found:** {len(df_netflix):,}')

  # --- Performance Optimization ---
  sample_size = st.slider('Select Sample Size for NLP Analysis', min_value=1000, max_value=20000,value=3000)
  df_sample = df_netflix.sample(n=sample_size,random_state=42)

  if "pdf_data" not in st.session_state:
        st.session_state.pdf_data = None

  if st.button("Run PM Analysis"):
        with st.spinner("Analyzing text & crunching data..."):

            # --- 1. Basic Stats ---
            st.subheader("1. App Star Rating Distribution")
            rating_counts = df_sample['score'].value_counts().sort_index()

            fig1, ax1 = plt.subplots(figsize=(10, 4))
            # Create a viridis color map based on the number of bars
            colors1 = plt.cm.viridis(np.linspace(0, 1, len(rating_counts)))
            ax1.bar(rating_counts.index.astype(str), rating_counts.values, color=colors1, edgecolor='black')
            ax1.set_xlabel("Star Rating")
            ax1.set_ylabel("Number of Reviews")
            ax1.grid(axis='y', linestyle='--', alpha=0.7)
            st.pyplot(fig1)

            # --- 2. Sentiment Analysis ---
            df_sample['Sentiment_Score'] = df_sample['content'].astype(str).apply(lambda x: TextBlob(x).sentiment.polarity)
            df_sample['Sentiment_Label'] = df_sample['Sentiment_Score'].apply(
                lambda x: 'Positive' if x > 0.1 else ('Negative' if x < -0.1 else 'Neutral')
            )

            st.subheader("2. Qualitative Sentiment Distribution")
            sentiment_counts = df_sample['Sentiment_Label'].value_counts()

            fig2, ax2 = plt.subplots(figsize=(10, 4))
            # Create a viridis_r (reversed viridis) color map
            colors2 = plt.cm.viridis_r(np.linspace(0, 1, len(sentiment_counts)))
            ax2.bar(sentiment_counts.index, sentiment_counts.values, color=colors2, edgecolor='black')
            ax2.set_ylabel("Number of Reviews")
            ax2.grid(axis='y', linestyle='--', alpha=0.7)
            st.pyplot(fig2)

            # --- 3. Topic Modeling (LDA) ---
            st.subheader("3. Topic Modeling (Latent Dirichlet Allocation)")
            custom_stop_words = list(CountVectorizer(stop_words='english').get_stop_words()) + ['app', 'netflix', 'movie', 'movies', 'show', 'shows', 'watch', 'watching', 'good', 'love', 'great']
            vectorizer = CountVectorizer(stop_words=custom_stop_words, max_df=0.85, min_df=5)
            dtm = vectorizer.fit_transform(df_sample['content'].astype(str))

            num_topics = 4
            lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
            lda.fit(dtm)

            topic_words = {}
            for index, topic in enumerate(lda.components_):
                top_words = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-6:]]
                topic_words[f"Theme {index+1}"] = ", ".join(top_words)
                st.write(f"**Theme {index+1} Keywords:** {topic_words[f'Theme {index+1}']}")

            topic_results = lda.transform(dtm)
            df_sample['Dominant_Topic'] = topic_results.argmax(axis=1) + 1

            # --- 4. Word Cloud ---
            st.subheader("4. Review Word Cloud")
            text_data = " ".join(df_sample['content'].astype(str))
            wordcloud = WordCloud(width=800, height=300, background_color='white', colormap='Reds').generate(text_data)
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.imshow(wordcloud, interpolation='bilinear')
            ax.axis("off")
            st.pyplot(fig)

            # --- 5. Generate Product Recommendations ---
            st.subheader("5. Actionable Product Recommendations")
            recs = []

            for i in range(1, num_topics + 1):
                topic_df = df_sample[df_sample['Dominant_Topic'] == i]
                avg_sent = topic_df['Sentiment_Score'].mean()
                avg_star = topic_df['score'].mean()
                keywords = topic_words[f"Theme {i}"]

                if avg_sent > 0.15:
                    action = "Promote & Maintain: High satisfaction. Leverage in App Store screenshots."
                elif avg_sent < 0.0:
                    action = "High Priority Fix: Investigate bugs/UX friction related to these terms."
                else:
                    action = "Needs Discovery: Conduct user interviews to clarify mixed feedback."

                recs.append({
                    "Theme": f"Theme {i}",
                    "Keywords": keywords,
                    "Avg Sentiment": str(round(avg_sent, 2)),
                    "Avg Star Rating": str(round(avg_star, 1)),
                    "PM Recommendation": action
                })

            rec_df = pd.DataFrame(recs)
            st.dataframe(rec_df)

            # --- 6. Generate PDF and Save to Memory ---
            def create_pdf(dataframe):
                pdf = FPDF()
                pdf.add_page()
                pdf.set_font("Arial", 'B', 16)
                pdf.set_text_color(229, 9, 20) # Netflix Red
                pdf.cell(0, 10, "Netflix Product Manager Recommendations", ln=True, align='C')
                pdf.ln(10)

                for index, row in dataframe.iterrows():
                    pdf.set_text_color(0, 0, 0)
                    pdf.set_font("Arial", 'B', 12)
                    pdf.cell(0, 8, f"{row['Theme']} - Keywords: {row['Keywords']}", ln=True)

                    pdf.set_font("Arial", '', 11)
                    pdf.set_text_color(80, 80, 80)
                    pdf.cell(0, 6, f"Avg Sentiment: {row['Avg Sentiment']} | Avg Star Rating: {row['Avg Star Rating']}", ln=True)

                    pdf.set_font("Arial", 'I', 11)
                    pdf.set_text_color(229, 9, 20)
                    pdf.multi_cell(0, 6, f"Action: {row['PM Recommendation']}")
                    pdf.ln(5)

                return pdf.output(dest='S').encode('latin-1')

            # Save the compiled PDF into Streamlit's session state
            st.session_state.pdf_data = create_pdf(rec_df)

  # --- 7. Download Button (OUTSIDE the run block) ---
  if st.session_state.pdf_data is not None:
      st.write("### Export Insights")
      st.download_button(
          label="📄 Download PM Recommendations as PDF",
          data=st.session_state.pdf_data,
          file_name='netflix_pm_insights.pdf',
          mime='application/pdf'
      )


Overwriting app.py


In [ ]:
from pyngrok import ngrok
import os
from google.colab import userdata

ngrok.kill()

ngrok_auth_token = userdata.get('ngrok_token')
ngrok.set_auth_token(ngrok_auth_token)

public_url = ngrok.connect(8501,proto='http')
print('ReviewMiner Dashboard is live at: ', {public_url})

!streamlit run app.py --server.port 8501 &

ReviewMiner Dashboard is live at:  {<NgrokTunnel: "https://14b4-34-74-135-125.ngrok-free.app" -> "http://localhost:8501">}


2026-05-12 09:08:06.532 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.74.135.125:8501

